# 18 - DIgSILENT PowerFactory Co-Simulation & Model Setup

This tutorial explains how to connect **DIgSILENT PowerFactory** to an `energysim` multi-energy co-simulation canvas.

---

## 1. Overview & Requirements

DIgSILENT PowerFactory is an enterprise power system analysis suite widely used by transmission and distribution system operators (TSOs and DSOs), utilities, and industrial microgrid operators.

`energysim` integrates with PowerFactory via its official **Python API (2020+)**, executing balanced or unbalanced AC/DC load-flow (`ComLdf`), dynamic RMS simulations, or short-circuit calculations at each macro co-simulation step.

### Requirements:
- **PowerFactory Installation**: PowerFactory 2020, 2021, 2022, 2023, 2024, or later with a valid DIgSILENT license.
- **Python Version Matching**:
  PowerFactory ships pre-compiled C-extensions for specific Python versions. Your Python environment must match the 64-bit architecture and minor version supported by your PowerFactory build:
  - *PowerFactory 2022*: Python 3.9, 3.10
  - *PowerFactory 2023*: Python 3.10, 3.11
  - *PowerFactory 2024*: Python 3.11, 3.12
  - *PowerFactory 2025*: Python 3.12, 3.13

---

## 2. PowerFactory Configuration & Variable Syntax

### Locating the PowerFactory Python API (`pf_path`)
By default, the PowerFactory Python modules reside in your installation directory:
```
C:\Program Files\DIgSILENT\PowerFactory 2024\Python\3.11
```
You can pass this path directly to `energysim` via `pf_path=...` or add it to your system `PYTHONPATH`.

### Variable Naming Convention: `ElementName.attribute`
Inputs and outputs use dot-notation:
- **Inputs (Element Parameters)**:
  - `'Load1.plini'` : Active power setpoint of `Load1` in MW.
  - `'Load1.qlini'` : Reactive power setpoint in Mvar.
  - `'PV_Park.pgini'`: Active power dispatch of static generator in MW.
  - `'Grid_Breaker.on_off'`: Breaker switch state (0 or 1).
- **Outputs (Calculation Results)**:
  - `'Bus1.m:u'` : Voltage magnitude at `Bus1` (p.u.).
  - `'Bus1.m:phiu'` : Voltage angle at `Bus1` (degrees).
  - `'Line_1_2.m:loading'`: Thermal branch loading (%).
  - `'Trafo_1.m:loading'`: Substation transformer loading (%).


In [ ]:
import os
import sys
from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from energysim import world

# Check whether the PowerFactory Python API is available
try:
    import powerfactory
    HAS_POWERFACTORY = True
except ImportError:
    HAS_POWERFACTORY = False

print(f"PowerFactory API importable : {HAS_POWERFACTORY}")
print(f"Mode                        : {'LIVE POWERFACTORY API' if HAS_POWERFACTORY else 'EDUCATIONAL SIMULATION MOCK'}")


## 3. Co-Simulation Setup

In this example:
1. An industrial feeder model contains grid slack, an industrial park load, and a large ground-mounted solar farm.
2. A solar irradiance signal drives the solar farm's active generation.
3. A shift-work schedule drives the factory active power demand.
4. `energysim` steps the power flow, converting units automatically (e.g. kW to MW) and recording voltages and loadings.


In [ ]:
sim = world(start_time=0, stop_time=3600 * 12, t_macro=900, power_unit="kW")

# Dynamic generation and load signals
sim.add_signal("solar_gen_kw", lambda t: max(0.0, 3500.0 * np.sin(np.pi * t / (12*3600))), step_size=900, units={"y": "kW"})
sim.add_signal("factory_load_kw", lambda t: 2200.0 + 600.0 * np.cos(2 * np.pi * t / (12*3600)), step_size=900, units={"y": "kW"})

if HAS_POWERFACTORY:
    # Live PowerFactory Adapter
    sim.add_powerfactory(
        sim_name="pf_grid",
        project_name="Industrial_Feeder",
        inputs=["Solar_Farm.pgini", "Industrial_Park.plini"],
        outputs=["Substation_Bus.m:u", "Feeder_Bus.m:u", "Main_Trafo.m:loading", "Line_1.m:loading"],
        mode="ac",
        step_size=900
    )
else:
    # Transparent educational mock of the IEEE / Industrial Feeder load flow
    from energysim.base import SimulatorAdapter
    class MockPowerFactoryAdapter(SimulatorAdapter):
        def __init__(self):
            self.inputs = ["Solar_Farm.pgini", "Industrial_Park.plini"]
            self.outputs = ["Substation_Bus.m:u", "Feeder_Bus.m:u", "Main_Trafo.m:loading", "Line_1.m:loading"]
            self.p_solar_mw = 0.0
            self.p_load_mw = 2.0
            self.v_sub = 1.000
            self.v_feeder = 0.995
            self.trafo_load = 50.0
            self.line_load = 40.0

        def set_value(self, variables, values):
            for v, val in zip(variables, values):
                if "pgini" in v: self.p_solar_mw = float(val) / 1000.0 if float(val) > 10.0 else float(val)
                elif "plini" in v: self.p_load_mw = float(val) / 1000.0 if float(val) > 10.0 else float(val)

        def get_value(self, variables, time=None):
            # Calculate linearized power flow approximations
            net_mw = self.p_load_mw - self.p_solar_mw
            v_drop = 0.012 * net_mw
            self.v_feeder = 1.00 - v_drop
            self.trafo_load = min(120.0, max(10.0, abs(net_mw) / 5.0 * 100.0))
            self.line_load = min(120.0, max(8.0, abs(net_mw) / 4.0 * 100.0))
            store = {
                "Substation_Bus.m:u": self.v_sub,
                "Feeder_Bus.m:u": self.v_feeder,
                "Main_Trafo.m:loading": self.trafo_load,
                "Line_1.m:loading": self.line_load
            }
            return [store[v] for v in variables]

        def step(self, time, step_size=900.0):
            pass

    sim.add_model("pf_grid", MockPowerFactoryAdapter())

# Route connections (automatic kW -> MW conversion)
sim.add_connections({
    "solar_gen_kw.y": "pf_grid.Solar_Farm.pgini",
    "factory_load_kw.y": "pf_grid.Industrial_Park.plini"
})

# Run co-simulation
res = sim.simulate(pbar=False)
df = res.to_frame()

print("PowerFactory co-simulation completed!")
display(df[["solar_gen_kw.y", "factory_load_kw.y", "pf_grid.Feeder_Bus.m:u", "pf_grid.Main_Trafo.m:loading"]].head())


## 4. Visualizing Grid Dynamics

Let's examine the feeder voltage profile and transformer thermal loading as solar generation ramps up:


In [ ]:
hours = df.index / 3600.0

fig, axes = plt.subplots(3, 1, figsize=(9, 7.5), sharex=True)

axes[0].plot(hours, df["solar_gen_kw.y"], label="Solar Farm Generation [kW]", color="tab:orange", lw=1.8)
axes[0].plot(hours, df["factory_load_kw.y"], label="Industrial Load [kW]", color="tab:blue", lw=1.8)
axes[0].set_ylabel("Power [kW]")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_title("PowerFactory Co-Simulation: Solar Ramp & Voltage Regulation", fontsize=11)

axes[1].plot(hours, df["pf_grid.Feeder_Bus.m:u"], label="Feeder Bus Voltage [pu]", color="tab:green", lw=2)
axes[1].axhline(1.05, color="red", ls="--", lw=0.9, label="Overvoltage Limit (+5%)")
axes[1].axhline(0.95, color="red", ls="--", lw=0.9, label="Undervoltage Limit (-5%)")
axes[1].set_ylabel("Voltage [p.u.]")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(hours, df["pf_grid.Main_Trafo.m:loading"], label="Substation Transformer Loading [%]", color="tab:purple", lw=1.8)
axes[2].axhline(100.0, color="k", ls="--", lw=0.9, label="Nominal Rating (100%)")
axes[2].set_ylabel("Loading [%]")
axes[2].set_xlabel("Time [hours]")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 5. Summary & Best Practices

1. **Keep Python & PowerFactory minor versions matched** (e.g. 64-bit Python 3.11 with PF 2024).
2. **Use dot notation** (`ElementName.attribute`) for clean, transparent variable targeting.
3. **Take advantage of automatic unit scaling**: declare `units={"y": "kW"}` in signals and `energysim` scales them to PowerFactory's `MW` parameters automatically.
4. **Multi-carrier workflows**: combine PowerFactory electrical grids with thermal networks, hydrogen electrolyzers, and gas pipelines on the same `energysim.world` canvas!
